# Training Series: Introduction to Discontinuous Galerkin Methods for Flow Problems <br> - Hands-On Worksheets

In [1]:
//#r "../BoSSSpad/BoSSSpad.dll"
#r "C:\Program Files (x86)\FDY\BoSSS\bin\Release\net6.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.LinSolvers;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Platform.LinAlg;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

## Hands-On 3: Poisson Equation and the Symmetric Interior Penalty Flux 

## Problem statement: 1D Poisson Equation

We consider the following problem:

$$\Delta u = g_{\Omega} = 2,\quad -1<x<1,\quad u(-1)=u(1)=0.$$

The solution is $u_{ex}(x) = 1 - x^2$. Since this is quadratic, we can represent it **exactly** in a DG space of order 2.

We recall the corresponding global variational formulation: find $u \in \mathbb{P}_k(\mathfrak{K}_h)$ such that 

$$\oint_{\Gamma} \hat F (u^-, u^+, \vec{n}_{\Gamma}) \llbracket{v\rrbracket}\ \textrm{d}A  
- \int_{\Omega} \vec{f}(u) \cdot \nabla_h v \ \textrm{d}V = \int_{\Omega} g_{\Omega} v \ \textrm{d}V$$

## Counterexample: A naive approach using the central flux

We notice that $\Delta{u} = \nabla \cdot (\nabla{u})$, therefore we use in our generic. time-dependent equation $\text{div}(\vec{f}(u))=g_{\Omega}$

$$\begin{align*}
\vec{f} &= - \nabla{u}\\
\hat{F} &= - \{ \nabla{u} \}\cdot n_{\Gamma}.
\end{align*}$$

Inserting that into the global variational formulation yields

$$\underbrace{-\oint_{\Gamma} \{ \nabla{u} \}\cdot n_{\Gamma} \llbracket{v\rrbracket}\ \textrm{d}A  
+ \int_{\Omega} \nabla{u} \cdot \nabla_h v \ \textrm{d}V}_{:= a_{\text{naive}}(u,v)} = \int_{\Omega} g_{\Omega} v \ \textrm{d}V$$

First, we need a class in which the integrands are defined.
This also includes some technical aspects like the *TermActivationFlags*.

In [2]:
public class NaivePoissonFlux :
        BoSSS.Foundation.IEdgeForm,   // edge integrals
        BoSSS.Foundation.IVolumeForm // volume integrals
{
    /// We do not use parameters (e.g. variable viscosity, ...)
    /// at this point: so this can be null
    public IList<string> ParameterOrdering { 
        get { return new string[0]; } 
    } 
    /// but we have one argument variable, $u$ (our trial function)
    public IList<String> ArgumentOrdering { 
        get { return new string[] { "u" }; } 
    }
    /// The \code{TermActivationFlags} tell \BoSSS which kind of terms, 
    /// i.e. products of u, v, \nabla u, and \nabla v
    /// the VolumeForm(...) actually contains.
    /// This additional information helps to improve the performance.
    public TermActivationFlags VolTerms {
        get {
            return TermActivationFlags.GradUxGradV;
        }
    }
    /// activation flags for the 'InnerEdgeForm(...)'
    public TermActivationFlags InnerEdgeTerms {
        get {
            return (TermActivationFlags.GradUxV);
        }
    }
    /// and 'BoundaryEdgeForm(...)'
    public TermActivationFlags BoundaryEdgeTerms {
       get {
           return TermActivationFlags.AllOn;
        }
    }

    
    /// The following functions cover the actual math.
    /// For any discretization of the Laplace operator, we have to specify:
    /// 
    /// - a volume integrand,
    /// - an edge integrand for inner edges, i.e. on $\Gamma_i$,
    /// - an edge integrand for boundary edges, i.e. on $\partial \Omega$.
    /// 
    /// The integrand for the volume integral:
    public double VolumeForm(ref CommonParamsVol cpv, 
           double[] U, double[,] GradU, // the trial-function u (i.e. the function we search for) and its gradient
           double V, double[] GradV     // the test function; 
           ) {
 
        double acc = 0.0;
        for(int d = 0; d < cpv.D; d++)
            acc += GradU[0, d] * GradV[d];
        return acc;
    }

    /// The integrand for the integral on the inner edges,
    /// 
    ///   -( {\nabla u} [[v]]) \cdot \vec{n}_{\Gamma} 
    /// 
    public double InnerEdgeForm(ref CommonParams inp, 
        double[] U_IN, double[] U_OT, double[,] GradU_IN, double[,] GradU_OT, 
        double V_IN, double V_OT, double[] GradV_IN, double[] GradV_OT) {
 
        double Acc = 0.0;
        for(int d = 0; d < inp.D; d++) { // loop over vector components 
            Acc -= 0.5 * (GradU_IN[0, d] + GradU_OT[0, d])*(V_IN - V_OT)
                       * inp.Normal[d];
        }
        return Acc;
    } 

    /// The integrand on boundary edges, i.e. on $\partial \Omega$, is
    ///  
    ///   -( {\nabla u} [[v]]) \cdot \vec{n}_{\Gamma} 
    /// 
    /// For the boundary we have to consider the special definition for 
    /// the mean-value operator ${-}$ and the jump operator $[[-]]$ on the boundary.
    public double BoundaryEdgeForm(ref CommonParamsBnd inp, 
        double[] U_IN, double[,] GradU_IN, double V_IN, double[] GradV_IN) {
        
        double Acc = 0.0;
        for(int d = 0; d < inp.D; d++) { // loop over vector components 
            Acc -= (GradU_IN[0, d])*(V_IN - 0.0) * inp.Normal[d];
        }
        return Acc;
    }
}

As usual, we have to set up a grid, a basis and a right-hand-side.

In [3]:
var grd1D                     = Grid1D.LineGrid(GenericBlas.Linspace(-1,1,4));  // 3 cells
var DGBasisOn1D               = new Basis(grd1D, 2);
var RHS                       = new SinglePhaseField(DGBasisOn1D, "RHS");
RHS.ProjectField((double x) => 2);

We will see that the discretization is *consistent*, but not stable.
* *Consistency*: $u_{\text{ex}}$ is a (exact) solution $\leftrightarrow$ $a_{\text{naive}}(u_{\text{ex}}, v) = \int_{\Omega} g_{\Omega} v \ \textrm{d}V $ should hold for every test function $v$.
* *Stability* will guarantee the uniqueness of the numerical solution.


The naive form is completely insensitive to jumps between cells. We now want to calculate the residual after inserting the exact solution as well as a wrong solution. 
The implementation of the exact solution:

In [4]:
var u_ex         = new SinglePhaseField(DGBasisOn1D, "u_{exact}");
u_ex.ProjectField((double x) => 1.0 - x*x);

For the wrong solution we add a constant $c$ in cell $K_1$, $u_{\text{wrong}}(x) = u_{\text{ex}}(x) + 1_{K_1}(x) \cdot c$

In [5]:
var u_wrong      = new SinglePhaseField(DGBasisOn1D, "u_{wrong}");
u_wrong.ProjectField((double x) => 1.0 - x*x);  // exact solution
int jCell = 1;
double c = 1.0;
double ujMean = u_wrong.GetMeanValue(jCell);
ujMean += c;
u_wrong.SetMeanValue(jCell, ujMean);

Let's plot both solutions

In [6]:
var gp = new Gnuplot();
int upsampling = 20;

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


In [7]:
gp.PlotField(u_ex, new PlotFormat(lineColor: LineColors.Blue, lineWidth: 2), upsampling);
gp.PlotField(u_wrong, new PlotFormat(lineColor: LineColors.Red, lineWidth: 2), upsampling);

In [8]:
gp.PlotNow()

Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 0.2 
 
 
 
 
 0.4 
 
 
 
 
 0.6 
 
 
 
 
 0.8 
 
 
 
 
 1 
 
 
 
 
 1.2 
 
 
 
 
 1.4 
 
 
 
 
 1.6 
 
 
 
 
 1.8 
 
 
 
 
 2 
 
 
 
 
 -1.5 
 
 
 
 
 -1 
 
 
 
 
 -0.5 
 
 
 
 
 0 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 
 
 
 
 uexact 
 
 
 
 
 u exact 
 
 
 
	<path stroke='rgb( 0, 0, 255)' d='M716.2,34.7 L758.4,34.7 M198.1,564.0 L208.2,545.1 L218.4,526.9 L228.5,509.4 L238.6,492.6 L248.7,476.4
 L258.8,460.9 L269.0,446.1 L279.1,431.9 L289.2,418.5 L299.3,405.7 L309.4,393.5 L319.6,382.1 L329.7,371.3
 L339.8,361.2 L349.9,351.8 L360.1,343.0 L370.2,334.9 L380.3,327.5 L390.4,320.8 L400.5,314.7 L410.7,309.3
 L420.8,304.6 L430.9,300.5 L441.0,297.2 L451.1,294.5 L461.3,292.5 L471.4,291.1 L481.5,290.4 L491.6,290.4
 L501.7,291.1 L511.9,292.5 L522.0,294.5 L532.1,297.2 L542.2,300.5 L552.3,304.6 L562.5,309.3 L572.6,314.7
 L582.7,320.8 L592.8,327.5 L602.9,334.9 L613.1,343.0 L623.2,351.8 L633.3,361.2 L643.4,371.3 L653.6,382.1
 L663.7,393.5 L673.8,405.7 L683.9,418.5 L694.0,431.9 L704.2,446.1 L714.3,460.9 L724.4,476.4 L734.5,492.6
 L744.6,509.4 L754.8,526.9 L764.9,545.1 L775.0,564.0 '/> 
 
 uwrong 
 
 
 u wrong 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M716.2,52.7 L758.4,52.7 M198.1,564.0 L208.2,545.1 L218.4,526.9 L228.5,509.4 L238.6,492.6 L248.7,476.4
 L258.8,460.9 L269.0,446.1 L279.1,431.9 L289.2,418.5 L299.3,405.7 L309.4,393.5 L319.6,382.1 L329.7,371.3
 L339.8,361.2 L349.9,351.8 L360.1,343.0 L370.2,334.9 L380.3,327.5 L390.4,320.8 L390.4,47.1 L400.5,41.0
 L410.7,35.7 L420.8,30.9 L430.9,26.9 L441.0,23.5 L451.1,20.8 L461.3,18.8 L471.4,17.5 L481.5,16.8
 L491.6,16.8 L501.7,17.5 L511.9,18.8 L522.0,20.8 L532.1,23.5 L542.2,26.9 L552.3,30.9 L562.5,35.7
 L572.6,41.0 L582.7,47.1 L582.7,320.8 L592.8,327.5 L602.9,334.9 L613.1,343.0 L623.2,351.8 L633.3,361.2
 L643.4,371.3 L653.6,382.1 L663.7,393.5 L673.8,405.7 L683.9,418.5 L694.0,431.9 L704.2,446.1 L714.3,460.9
 L724.4,476.4 L734.5,492.6 L744.6,509.4 L754.8,526.9 L764.9,545.1 L775.0,564.0 '/>

Evaluating the Laplace operator using the different solutions:

In [9]:
var i_NaiveLaplace              = new NaivePoissonFlux();
var Operator_NaiveLaplace       = i_NaiveLaplace.Operator();
var Residual     = new SinglePhaseField(DGBasisOn1D,"Resi1");
var ResidualNorm = new List<double>();
foreach(var u in new DGField[] {u_ex, u_wrong}) {
    Residual.Clear();
    Operator_NaiveLaplace.Evaluate(u, Residual);  // evaluate
    Residual.Acc(-1.0, RHS);    
    double ResiNorm = Residual.L2Norm();
    ResidualNorm.Add(ResiNorm);
    Console.WriteLine("Residual for " + u.Identification + " = " + ResiNorm);  
}

Residual for u_{exact} = 7.179858070600463E-14
Residual for u_{wrong} = 7.179858070600463E-14


So the above discretization allows infinitely many wrong solutions and is thus not stable resp. unstable. One possibility for a discretization that is *consistent* **and** *stable* will be implemented in the next section.

## Symmteric Interior Penalty

### Problem statement

We consider the 2D Poisson problem:
$$ \Delta u = f(x,y) $$

where $f(x,y) \neq 0$ is an arbitrary function of $x$ and/or $y$.
Within this exercise, we are going to investigate the Symmetric Interior Penalty discretization method (SIP) for the Laplace operator

$$a_{\text{sip}}(u,v)
= \int_{\Omega} \underbrace{\nabla u \cdot \nabla v}_{\text{Volume\ term}}dV
  - \oint_{\Gamma \setminus \Gamma_{N }} \underbrace{
        \{\nabla u\} \cdot n_{\Gamma} \llbracket v \rrbracket
     }_{\text{consistency term}} + \underbrace{
        \{\nabla v\} \cdot \vec{n}_{\Gamma} \llbracket u \rrbracket
     }_{\text{symmetry term}} dA
  + \oint_{\Gamma \setminus \Gamma_{N}} \underbrace{
       \eta \llbracket u \rrbracket \llbracket v \rrbracket
    }_{\text{penalty term}} dA$$

The use of these fluxes includes a penalty term stabilizing the DG-discretization for the Laplace operator.

### Implementation of the SIP fluxes

As before, we need first a class in which the integrands are defined.

In [10]:
public class SipLaplace :
        BoSSS.Foundation.IEdgeForm,   // edge integrals
        BoSSS.Foundation.IVolumeForm, // volume integrals
        IEquationComponentCoefficient // update of coefficients (e.g. length scales) required for penalty parameters 
{
    /// We do not use parameters (e.g. variable viscosity, ...)
    /// at this point: so this can be null
    public IList<string> ParameterOrdering { 
        get { return new string[0]; } 
    } 
    /// but we have one argument variable, $u$ (our trial function)
    public IList<String> ArgumentOrdering { 
        get { return new string[] { "u" }; } 
    }
    /// The \code{TermActivationFlags} tell \BoSSS which kind of terms, 
    /// i.e. products of u, v, \nabla u, and \nabla v
    /// the VolumeForm(...) actually contains.
    /// This additional information helps to improve the performance.
    public TermActivationFlags VolTerms {
        get {
            return TermActivationFlags.GradUxGradV;
        }
    }
    /// activation flags for the 'InnerEdgeForm(...)'
    public TermActivationFlags InnerEdgeTerms {
        get {
            return (TermActivationFlags.AllOn);
            // if we do not care about performance, we can activate all terms.
        }
    }
    public TermActivationFlags BoundaryEdgeTerms {
       get {
           return TermActivationFlags.AllOn;
        }
    }

    /// For the computation of the penalty factor $\eta$,
    /// we require some length scale for each cell and 
    /// the polynomial degree of the DG approximation.
    MultidimensionalArray cj;

    double penalty_base; // base factor must be scaled by polynomial degree  

    /// The additional scaling of the penalty by polynomial degree 
    /// and in depencence of geometry can be obtained through 
    /// impmenting the \code{IEquationComponentCoefficient} interface:
    public void CoefficientUpdate(CoefficientSet cs, int[] DomainDGdeg, int TestDGdeg) {
        int D = cs.GrdDat.SpatialDimension;
        double _D = D;
        double _p = DomainDGdeg.Max();

        double penalty_deg_tri = (_p + 1) * (_p + _D) / _D; // formula for triangles/tetras
        double penalty_deg_sqr = (_p + 1.0) * (_p + 1.0); // formula for squares/cubes

        penalty_base = Math.Max(penalty_deg_tri, penalty_deg_sqr); // the conservative choice
        //Console.WriteLine("Setting penalty base factor for deg " + _p + " to " + penalty_base);

        cj = ((GridData)(cs.GrdDat)).Cells.cj; // inverse length scale (dimension is one-over-length, resp. area over volume)
    }     
    
            
    /// The safety factor for the penalty factor should be in the order of 1.
    /// A very large penalty factor increases the condition number of the system, but without affecting stability.
    /// A very small penalty factor yields to an unstable discretization.
    public double PenaltySafety = 2.2; 

    /// The actual computation of the penalty factor, which should be used in the \code{InnerEdgeForm} and \code{BoundaryEdgeForm} functions.
    /// Hint: for the parameters \code{jCellIn}, \code{jCellOut} and \code{g}, take a look at \code{CommonParams} and \code{CommonParamsBnd}.
    double PenaltyFactor(int jCellIn, int jCellOut) {
        double cj_in         = cj[jCellIn];
        double eta           = penalty_base * cj_in * PenaltySafety;
        if(jCellOut >= 0) {
            double cj_out = cj[jCellOut];
            eta           = Math.Max(eta, penalty_base * cj_out * PenaltySafety);
        }
        
        return eta;
    }
    
    /// The following functions cover the actual math.
    /// For any discretization of the Laplace operator, we have to specify:
    /// 
    /// - a volume integrand,
    /// - an edge integrand for inner edges, i.e. on $\Gamma_i$,
    /// - an edge integrand for boundary edges, i.e. on $\partial \Omega$.
    /// 
    /// The integrand for the volume integral:
    public double VolumeForm(ref CommonParamsVol cpv, 
           double[] U, double[,] GradU, // the trial-function u (i.e. the function we search for) and its gradient
           double V, double[] GradV     // the test function; 
           ) {
 
        // == TODO == add the SIP volume form 
        double acc = 0.0;
        for(int d = 0; d < cpv.D; d++)
            acc += GradU[0, d] * GradV[d];
        return acc;
    }

    /// The integrand for the integral on the inner edges,
    /// 
    ///   -( {\nabla u} [[v]]) \cdot \vec{n}_{\Gamma} 
    ///   -( {\nabla v} [[u]]) \cdot \vec{n}_{\Gamma} 
    ///   + \eta [[u]] [[v]] :
    /// 
    public double InnerEdgeForm(ref CommonParams inp, 
        double[] U_IN, double[] U_OT, double[,] GradU_IN, double[,] GradU_OT, 
        double V_IN, double V_OT, double[] GradV_IN, double[] GradV_OT) {
 
        double eta = PenaltyFactor(inp.jCellIn, inp.jCellOut);
 
        double Acc = 0.0;
        // == TODO == add the consistency term : -({ \/u } [[ v ]])*Normal
        // == TODO == add the symmetry term: : -({ \/v } [[ u ]])*Normal
        // == TODO == add the penalty term: eta*[[u]]*[[v]]

        for(int d = 0; d < inp.D; d++) { // loop over vector components 
            // consistency term: -({ \/u } [[ v ]])*Normal
            // index d: spatial direction
            Acc -= 0.5 * (GradU_IN[0, d] + GradU_OT[0, d])*(V_IN - V_OT)
                       * inp.Normal[d];
 
            // symmetry term: -({ \/v } [[ u ]])*Normal
            Acc -= 0.5 * (GradV_IN[d] + GradV_OT[d])*(U_IN[0] - U_OT[0])
                       * inp.Normal[d];
        }
 
        // penalty term: eta*[[u]]*[[v]]
        Acc += eta*(U_IN[0] - U_OT[0])*(V_IN - V_OT);
        
        return Acc;
    } 

    /// The integrand on boundary edges, i.e. on $\partial \Omega$, is
    ///  
    ///   -( {\nabla u} [[v]]) \cdot \vec{n}_{\Gamma} 
    ///   -( {\nabla v} [[u]]) \cdot \vec{n}_{\Gamma} 
    ///   +  \eta [[u]]  [[v]] .
    /// 
    /// For the boundary we have to consider the special definition for 
    /// the mean-value operator ${-}$ and the jump operator $[[-]]$ on the boundary.
    public double BoundaryEdgeForm(ref CommonParamsBnd inp, 
        double[] U_IN, double[,] GradU_IN, double V_IN, double[] GradV_IN) {
 
        double eta = PenaltyFactor(inp.jCellIn, -1);
        
        double Acc = 0.0;
        // == TODO == add the consistency term: -({ \/u } [[ v ]])*Normale
        // == TODO == add the symmetry term: -({ \/v } [[ u ]])*Normale
        // == TODO == add the penalty term: eta*[[u]]*[[v]]

        for(int d = 0; d < inp.D; d++) { // loop over vector components 
            // consistency term: -({ \/u } [[ v ]])*Normale
            // index d: spatial direction
            Acc -= (GradU_IN[0, d])*(V_IN - 0.0) * inp.Normal[d];
 
            // symmetry term: -({ \/v } [[ u ]])*Normale
            Acc -= (GradV_IN[d])*(U_IN[0]) * inp.Normal[d];
        }
 
        // penalty term: eta*[[u]]*[[v]]
        Acc += eta*(U_IN[0])*(V_IN - 0.0);
 
        return Acc;
    }
}

### Evaluation of the Poisson operator in 1D

As usual, we have to set up a new grid (with 9 cells this time), a corresponding basis and right-hand-side.

In [11]:
var grd1D                     = Grid1D.LineGrid(GenericBlas.Linspace(-1,1,10));
var DGBasisOn1D               = new Basis(grd1D, 2);
var RHS                       = new SinglePhaseField(DGBasisOn1D, "RHS");
RHS.ProjectField((double x) => 2);

The exact solution on the new basis is:

In [12]:
var u_ex         = new SinglePhaseField(DGBasisOn1D, "$u_{exact}$");
u_ex.ProjectField((double x) => 1.0 - x*x);

The implementation of a spurious, i.e. a wrong solution; we take the exact solution and add random values in each cell:

In [13]:
var u_wrong      = new SinglePhaseField(DGBasisOn1D, "$u_{wrong}$");
u_wrong.ProjectField((double x) => 1.0 - x*x);
Random R         = new Random();
for(int j = 0; j < grd1D.GridData.Cells.NoOfLocalUpdatedCells; j++){
    double ujMean = u_wrong.GetMeanValue(j);
    ujMean += R.NextDouble();
    u_wrong.SetMeanValue(j, ujMean);
}

We evaluate the Laplace operator using the different solutions:

In [14]:
var i_SipLaplace              = new SipLaplace();
var Operator_SipLaplace       = i_SipLaplace.Operator();
var Residual     = new SinglePhaseField(DGBasisOn1D,"Resi1");
var ResidualNorm = new List<double>();
foreach(var u in new DGField[] {u_ex, u_wrong}) {
    Residual.Clear();
    Operator_SipLaplace.Evaluate(u, Residual);  // evaluate
    Residual.Acc(-1.0, RHS);    
    double ResiNorm = Residual.L2Norm();
    ResidualNorm.Add(ResiNorm);
    Console.WriteLine("Residual for " + u.Identification + " = " + ResiNorm);  
}

Residual for $u_{exact}$ = 1.3693768325334959E-12
Residual for $u_{wrong}$ = 1108.5691497437053


Now we see that the residual for the wrong "solution" is actually greater than zero. The following code checks your result wihtin the nunit framework:

In [15]:
using NUnit.Framework;

In [16]:
Assert.LessOrEqual(ResidualNorm[0], 1e-10);
Assert.GreaterOrEqual(ResidualNorm[1], 1e-1);

### The penalty parameter of the SIP and stability in 2D

We define a two-dimensional grid:

In [17]:
var grd2D       = Grid2D.Cartesian2DGrid(GenericBlas.Linspace(-1,1,21), 
                                         GenericBlas.Linspace(-1,1,16));
var DGBasisOn2D = new Basis(grd2D, 5);
var Mapping2D   = new UnsetteledCoordinateMapping(DGBasisOn2D);

We consider the example 
$$
    -\Delta u = \pi^2 (a_x^2 + a_y^2)/4 \cos(a_x \pi x/2) \cos(a_y \pi y/2) 
      \text{ with } 
      (x,y) \in (-1,1)^2
$$
and $u = 0$ on the boundary.
The exact solution is $u_{exact}(x,y) = \cos(a_x \pi  x/2) \cos(a_x \pi y/2)$, where $a_x$ and $a_y$ must be odd numbers
to comply with homogeneous bounary condition.

In [18]:
double ax = 1.0; // must be an odd number to comply with homogeneous Dirichlet boundary condition
double ay = 3.0; // must be an odd number to comply with homogeneous Dirichlet boundary condition
Func<double[], double> exSol = 
        (X => Math.Cos(X[0]*ax*Math.PI*0.5)*Math.Cos(X[1]*ay*Math.PI*0.5));
Func<double[], double> exRhs = 
        (X => ((ax.Pow2() + ay.Pow2())/4.0)*Math.PI.Pow2()
             *Math.Cos( X[0]*ax*Math.PI*0.5 )*Math.Cos( X[1]*ay*Math.PI*0.5 )); // == - /\ exSol

SinglePhaseField RHS = new SinglePhaseField(DGBasisOn2D, "RHS");
RHS.ProjectField(exRhs);
double[] RHSvec = RHS.CoordinateVector.ToArray();

We check our discretization once more in 2D; the residual should be low,
but not exactly (resp. up to $10^{-12}$) since the solution is not 
polynomial and cannot be fulfilled exactly.

In [19]:
SinglePhaseField u = new SinglePhaseField(DGBasisOn2D,"u");
u.ProjectField(exSol);

var Matrix_SIP_sf = Operator_SipLaplace.ComputeMatrix(Mapping2D, null, Mapping2D);

SinglePhaseField Residual = new SinglePhaseField(DGBasisOn2D,"Residual");
Residual.Acc(1.0, RHS);
Matrix_SIP_sf.SpMV(-1.0, u.CoordinateVector, 1.0, Residual.CoordinateVector);
Console.WriteLine("Residual L2 norm: " + Residual.L2Norm());

Residual L2 norm: 0.02104929706821074


== TASK == Above, how could you decrease the Residual L2 norm?

We also check that the matrix is symmetric:

In [20]:
var checkMatrix = Matrix_SIP_sf - Matrix_SIP_sf.Transpose();
checkMatrix.InfNorm()

5.837641481321043E-10

In [21]:
Assert.LessOrEqual(checkMatrix.InfNorm(), 1e-8);

### Matrix properties for different penalty factors (BONUS)

We are going to choose the **PenaltySafety** for the **SipLaplace**
from the following list

In [22]:
double[] SFs = new double[] {0.001, 0.002, 0.01, 0.02, 0.1, 0.2, 1, 2, 10, 20, 100};

Now, we assemble the matrix of the SIP for different 
**PenaltySafety**-factors. We also try to solve the linear system
using an iterative method.

 As Matlab is called multiple times during this 
command, it can take some minutes until it is done.

In case that BoSSS can't find the path to matlab you will need to modify (if Matlab is installed) a file called MatlabConnectorConfig.json which you can find in your BoSSS binary under the following path: **...\.BoSSS\etc\MatlabConnectorConfig.json**
Inside the file add a path to your matlab.exe (e.g.): **{ "MatlabExecuteable": "C:\ProgramFiles\MATLAB\R2021a\bin\win64\MATLAB.exe", "Flav": "Matlab" }**

In [23]:
using System.IO;
using ilPSP.Connectors.Matlab;

In [24]:
int cnt     = 0;
var Results = new List<(double safetyFactor, double condNumber, int NoOfIterations, double L2errror, bool isDefinite)>();
Console.WriteLine("sf" + "\t" + "cond num"
                        + "\t" + "No. of iter"
                        + "\t" + "L2-error"
                        + "\t" + "is definite?");

bool useStoredData = false;
bool useStoredDataMatlab = true;
StreamReader StoredData = new StreamReader(File.OpenRead("PenaltyFactorStudy.txt"));
string line = StoredData.ReadLine();

foreach(double sf in SFs) {

    line = StoredData.ReadLine();
    var items = line.Split("\t");

    cnt++;
    i_SipLaplace.PenaltySafety    = sf;
    var Matrix_SIP_sf             = Operator_SipLaplace.ComputeMatrix(
                                    Mapping2D, null, Mapping2D);
    double condNo1                = (useStoredData || useStoredDataMatlab) ? Convert.ToDouble(items[1]) : Matrix_SIP_sf.condest();  
    bool definite                 = (useStoredData || useStoredDataMatlab) ? Convert.ToBoolean(items[4]) : Matrix_SIP_sf.IsDefinite();
 
    /// We solve the system 
    /// 
    ///     Matrix\_SIP\_sf \cdot u =  RHS
    /// 
    /// using a an iterative solver, the so-called conjugate gradient (CG) method.
    /// CG requires a positive definite matrix. 
    /// The function \code{Solve\_CG} returns the number of iterations.
    SinglePhaseField u = new SinglePhaseField(DGBasisOn2D,"u");
    u.InitRandom();
    int NoOfIter = useStoredData ? Convert.ToInt16(items[2]) : Matrix_SIP_sf.Solve_CG(u.CoordinateVector, RHSvec);
 
    SinglePhaseField Error = new SinglePhaseField(DGBasisOn2D,"Error");
    Error.ProjectField(exSol);
    Error.Acc(-1.0, u);
 
    double L2err = useStoredData ? Convert.ToDouble(items[3]) : u.L2Error(exSol);
 
    Console.WriteLine(sf + "\t" + condNo1.ToString("0.#E-00") 
                         + "\t" + NoOfIter 
                         + "\t" + L2err.ToString("0.#E-00") 
                         + "\t" + definite);
    Results.Add((sf, condNo1, NoOfIter, L2err, definite));
}

sf	cond num	No. of iter	L2-error	is definite?
0.001	8.1E04	13411	4.6E-07	False
0.002	8.1E04	14856	3.7E-07	False
0.01	7.9E04	21160	1.6E-06	False
0.02	7.9E04	23214	2.6E-07	False
0.1	8.5E05	23652	3.7E-06	False
0.2	4.5E04	4687	3.9E-07	False
1	3.5E05	1455	1.7E-06	True
2	7.9E05	2071	3E-06	True
10	4.3E06	4214	3.3E-05	True
20	8.6E06	6055	3.4E-05	True
100	4.3E07	11572	2.5E-04	True


In [25]:
foreach(var r in Results) {
    if(r.safetyFactor >= 1 && r.safetyFactor <= 20) {
        Assert.LessOrEqual(r.condNumber, 1e7); // cond No.
        Assert.LessOrEqual(r.NoOfIterations, 7000); // iter
        Assert.LessOrEqual(r.L2errror, 1e-4); // L2 err
        Assert.IsTrue(r.isDefinite); // definite   
    }
    if(r.Item1 <= 0.1) {
        Assert.IsFalse(r.isDefinite); // indefinite   
    }
}

Plot the number of conjugate gradient iterations versus the **PenaltySafety**.

In [26]:
var xValues = Results.Select(r => r.safetyFactor).ToArray();
var yValues = Results.Select(r => ((double)(r.NoOfIterations))).ToArray();

var plt = new Plot2Ddata();
plt.AddDataGroup(xValues, yValues);

/// A logarithmic scale is used for the horizontal axis.
plt.LogX = true;

/// Set Format
plt.dataGroups[0].Format =  new PlotFormat(lineColor: LineColors.Blue, 
                                           pointSize: 2, 
                                           dashType: DashTypes.DotDashed, 
                                           Style: Styles.LinesPoints, 
                                           pointType:PointTypes.OpenCircle);
// Show!
plt.PlotNow()

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 5000 
 
 
 
 
 10000 
 
 
 
 
 15000 
 
 
 
 
 20000 
 
 
 
 
 25000 
 
 
 
 
 10 -3 
 
 
 
 
 10 -2 
 
 
 
 
 10 -1 
 
 
 
 
 10 0 
 
 
 
 
 10 1 
 
 
 
 
 10 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 gnuplot_plot_1

### Convergence study, indefinite vs. definite.

We are going to solve the SIP-system for different grid resolutions,
comparing an insufficient penalty to a penalty which is large enough.

In [27]:
double[] Resolution = new double[] { 2, 4, 8, 16, 32, 64 };
List<double> L2Error_indef  = new List<double>();
List<double> L2Error_posdef = new List<double>();
int cnt = 0;
foreach(int Res in Resolution) {
    cnt++;
    //var grd2D = Grid2D.UnstructuredTriangleGrid(GenericBlas.Linspace(-1,1,(int)Res + 1), 
    //                                            GenericBlas.Linspace(-1,1,(int)Res + 1));
    var grd2D = Grid2D.Cartesian2DGrid(GenericBlas.Linspace(-1,1,(int)Res + 1), 
                                      GenericBlas.Linspace(-1,1,(int)Res + 1));
    var gdata2D = new GridData(grd2D);
    var DGBasisOn2D = new Basis(gdata2D, 2);
    var Mapping2D  = new UnsetteledCoordinateMapping(DGBasisOn2D);
 
    SinglePhaseField RHS = new SinglePhaseField(DGBasisOn2D, "RHS");
    RHS.ProjectField(exRhs);
    SinglePhaseField uEx = new SinglePhaseField(
           new Basis(gdata2D, DGBasisOn2D.Degree*2),
           "Error");
    uEx.ProjectField(exSol);
 
 
    i_SipLaplace.PenaltySafety    = 0.01; // == TODO == choose a insufficient small penalty parameter (indefinite matrix)
    var Matrix_SIP_indef          = Operator_SipLaplace.ComputeMatrix(
                                    Mapping2D,null,Mapping2D);

    SinglePhaseField u_indef = new SinglePhaseField(DGBasisOn2D,"u_indef");
    Matrix_SIP_indef.Solve_Direct(u_indef.CoordinateVector, 
                                  RHS.CoordinateVector);
    var Error_indef = uEx.CloneAs();
    Error_indef.AccLaidBack(-1.0, u_indef);
    L2Error_indef.Add(Error_indef.L2Norm());
 
    /// In order to have a positive definite system, we are
    /// using  PenaltySafety = 2!
    i_SipLaplace.PenaltySafety    = 2.0; // == TODO == choose a sufficient large penalty parameter (definite matrix)
    var Matrix_SIP_posdef         = Operator_SipLaplace.ComputeMatrix(
                                    Mapping2D, null, Mapping2D);

    SinglePhaseField u_posdef = new SinglePhaseField(DGBasisOn2D,"u_posdef");
    Matrix_SIP_posdef.Solve_Direct(u_posdef.CoordinateVector, 
                                   RHS.CoordinateVector);
    var Error_posdef = uEx.CloneAs();
    Error_posdef.AccLaidBack(-1.0, u_posdef);
    L2Error_posdef.Add(Error_posdef.L2Norm());
    
    //Tecplot("ConvStudy-" + cnt, uEx, u_posdef, u_indef); // activate this line for plotting!
    
    Console.WriteLine(L2Error_indef.Last().ToString("0.#E-00") 
                      + "\t" + L2Error_posdef.Last().ToString("0.#E-00"));
}

1.5E01	7.2E-01
4.1E-01	1.9E-01
3.1E-02	1.7E-02
5.8E-03	1.6E-03
8.4E-04	1.7E-04


Solve_Direct fail in fallback seq (#0): Linear Solver: High residual from direct solver ilPSP.LinSolvers.PARDISO.PARDISOSolver.
    L2 Norm of RHS:         24.674010999677737
    L2 Norm of Solution:    0.9999956066685374
    L2 Norm of Residual:    0.001948079993917064
    Relative Residual norm: 1.6453742685621442E-08
    Matrix Inf norm:        118397.37810045174



1.1E-04	2.1E-05


### Convergence Plot and Conclusions

The convergence plot should unveil that there is something wrong if the
penalty factor is set too low.
Unfortunately, **it does not**, so this is some kind of **anti-example**;
It is in this tutorial anyway **to illustrate the difficulties of numerical testing**.

The reason why the indefinite matrix still gives a solution convergence 
is very likely that the solver which is used in BoSSS is also (sometimes) capable
of solving singular or close-to-singular systems, i.e. systems without a unique solution.
In those cases, it selects a solution with a minimal solution norm. 
Since BoSSS uses an orthonormal basis the $L^2$ norm of the DG-Field is identical to the $l_2$ norm of the 
coordinate vector (Parseval's identity).
Therefore, the solver by chance adds additional stability which is not part of the (instable) discretization.

The error of the positive definite system, *Error\_posdef*, where the 
penalty is chosen sufficiently large converges with the expected 
rate.

In [28]:
var plt = new Plot2Ddata();
plt.AddDataGroup("indef mtx", Resolution, L2Error_indef);
plt.AddDataGroup("pos def mtx", Resolution, L2Error_posdef);

/// A double-logarithmic scale is used:
plt.LogX = true;
plt.LogY = true;

/// Set Format
plt.dataGroups[0].Format =  new PlotFormat(lineColor: LineColors.Red, 
                                           pointSize: 2, 
                                           dashType: DashTypes.DotDashed, 
                                           Style: Styles.LinesPoints, 
                                           pointType:PointTypes.OpenCircle);
plt.dataGroups[1].Format =  new PlotFormat("Blue-.o"); // altenatively, using MATLAB-like format strings
plt.dataGroups[1].Format.PointSize = 2;
// Show!
plt.PlotNow()

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 10 -5 
 
 
 
 
 10 -4 
 
 
 
 
 10 -3 
 
 
 
 
 10 -2 
 
 
 
 
 10 -1 
 
 
 
 
 10 0 
 
 
 
 
 10 1 
 
 
 
 
 10 2 
 
 
 
 
 10 0 
 
 
 
 
 10 1 
 
 
 
 
 10 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 indef mtx 
 
 
 indef mtx 
 
 
 
 
 
 
 
 
 
 
 
 
 pos def mtx 
 
 
 pos def mtx

Finally, we are going to compute the convergence rate of the SIP
discretization.

We compute the slope of the log-log plot:

In [29]:
double dk = Resolution.LogLogRegressionSlope(L2Error_posdef);
dk

-3.1159715512848996

In [30]:
Assert.LessOrEqual(dk, -2.9);